# Mixing toys, decay-time acceptance and bin-flip

Toys of $D^0 \to K^0_S \pi^+ \pi^-$ with $D^0$–$\bar{D}^0$ mixing, generated from
the Belle 2018 amplitude model, as `macros/d02kspipi_toys.py` of tfa_examples (which
used the binned Belle amplitude). A decay-time acceptance is applied, $x$ and $y$ are
fitted, and the bin-flip ratios $R_b(t)$ and the strong-phase parameters $c_b, s_b$
are computed in the BaBar 2008 equal-$\Delta\delta_D$ binning.

To have a visible effect with a small sample, the mixing parameters are exaggerated:
$x = y = 0.05$ instead of the world averages $x \approx 0.4\%$, $y \approx 0.6\%$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import sympy as sp
from IPython.display import Math
from tensorwaves.data import IntensityDistributionGenerator
from tensorwaves.data.rng import NumpyUniformRNG
from tensorwaves.estimator import UnbinnedNLL
from tensorwaves.function.sympy import create_parametrized_function
from tensorwaves.optimizer import Minuit2

from aman_examples.acceptance import apply_efficiency, decay_time_efficiency
from aman_examples.binflip import bin_flip_ratios, binflip_bin, strong_phase_parameters
from aman_examples.d02kspipi import formulate_model
from aman_examples.mixing import amplitude_transformer, formulate_mixing_model
from aman_examples.phasespace import DalitzDecayTimeGenerator

## Model

The time-dependent rate is a function of the $D^0$ and $\bar{D}^0$ amplitudes
$A_f$, $\bar{A}_f$ and of the decay time $t$ (in units of $\tau$). It is written
relative to $e^{-t/\tau}$, because the decay times are generated from that
exponential (`DalitzDecayTimeGenerator`). The amplitudes are computed once per event
by a data transformer, so that the rate only depends on $x$, $y$ and $q/p$.

In [ ]:
dalitz_model = formulate_model()
mixing = formulate_mixing_model(x=0.05, y=0.05)
Math(sp.latex(mixing.intensity))

In [ ]:
masses = [
    dalitz_model.parameter_defaults[dalitz_model.masses[k]]
    for k in ("pi", "KS", "pi", "D")
]
transformer = amplitude_transformer(dalitz_model)
intensity = create_parametrized_function(
    mixing.intensity.doit(), mixing.parameter_defaults, backend="jax"
)
generator = DalitzDecayTimeGenerator(masses)
rng = NumpyUniformRNG(seed=0)
data = IntensityDistributionGenerator(
    generator, intensity, domain_transformer=transformer, bunch_size=200_000
).generate(100_000, rng)
phsp = generator.generate(2_000_000, rng)

## Decay-time acceptance

The efficiency $\varepsilon(t) = \left((t-t_0)^a / (b + (t-t_0)^n)\right)^m$ is
applied by accept-reject. With $a = n$ it rises from $t_0$ to a plateau, like a
trigger or selection turn-on; for $a > n$ it grows without bound and most events
would be rejected. The same efficiency is applied to the normalisation sample,
so that the fit below accounts for it.

In [ ]:
acceptance = {
    "t0": 0.2,
    "a": 2.0,
    "b": 0.3,
    "m": 1.0,
    "n": 2.0,
}  # turn-on, plateau at 1
data_acc = apply_efficiency(data, decay_time_efficiency(data["t"], **acceptance), rng=1)
phsp_acc = apply_efficiency(phsp, decay_time_efficiency(phsp["t"], **acceptance), rng=2)
print(f"{len(data_acc['t'])} of {len(data['t'])} events kept")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), tight_layout=True)
bins = np.linspace(0, 8, 81)
ax1.hist(data["t"], bins=bins, histtype="step", label="generated")
ax1.hist(data_acc["t"], bins=bins, histtype="step", label="after acceptance")
ax1.set_xlabel(r"$t/\tau$")
ax1.set_yscale("log")
ax1.legend()
ax2.hist2d(data_acc["m2_kspim"], data_acc["m2_kspip"], bins=100, cmin=1)
ax2.set_xlabel(r"$m^2(K^0_S\pi^-)$ [GeV$^2$]")
ax2.set_ylabel(r"$m^2(K^0_S\pi^+)$ [GeV$^2$]")
plt.show()

## Fit of $x$ and $y$

In [ ]:
estimator = UnbinnedNLL(
    intensity, transformer(data_acc), transformer(phsp_acc), backend="jax"
)
result = Minuit2().optimize(estimator, {"x": 0.0, "y": 0.0})
for name in ("x", "y"):
    value, error = result.parameter_values[name], result.parameter_errors[name]
    print(
        f"{name} = {value:.4f} ± {error:.4f}  (true 0.05, pull {(value - 0.05) / error:+.2f})"
    )

## Bin-flip ratios

$R_b(t) = N(+b)/N(-b)$ in decay-time bins with equal numbers of events, for the
toys and for the fitted model (the normalisation sample weighted by the rate).

In [ ]:
b_data = binflip_bin(data_acc["m2_kspim"], data_acc["m2_kspip"])
centers, ratio, error = bin_flip_ratios(data_acc["t"], b_data, n_time_bins=8)

intensity.update_parameters(result.parameter_values)
weights = intensity(transformer(phsp_acc))
b_phsp = binflip_bin(phsp_acc["m2_kspim"], phsp_acc["m2_kspip"])
edges = np.quantile(data_acc["t"], np.linspace(0, 1, 9))
time_bin = np.clip(np.digitize(phsp_acc["t"], edges) - 1, 0, 7)
expected = np.array([
    [
        weights[(b_phsp == b) & (time_bin == i)].sum()
        / weights[(b_phsp == -b) & (time_bin == i)].sum()
        for i in range(8)
    ]
    for b in range(1, 9)
])

fig, axes = plt.subplots(2, 4, figsize=(13, 5), sharex=True, tight_layout=True)
for b, ax in enumerate(axes.flat):
    ax.errorbar(centers, ratio[b], yerr=error[b], fmt="o", color="black", label="toys")
    ax.plot(centers, expected[b], label="fit")
    ax.set_title(f"$R_{b + 1}$")
for ax in axes[1]:
    ax.set_xlabel(r"$t/\tau$")
axes[0, 0].legend()
plt.show()

## Strong-phase parameters

$c_b + i s_b = \int_b A_f \bar{A}_f^* / \sqrt{\int_b |A_f|^2 \int_b |\bar{A}_f|^2}$
over bin $+b$, from the Belle model on events uniform in the Dalitz plot. The binning
was designed with the BaBar 2008 model so that the strong-phase difference varies by
equal $45^\\circ$ steps from bin to bin.

In [ ]:
uniform = generator.generate(2_000_000, rng)
amplitudes = transformer(uniform)
b_uniform = binflip_bin(uniform["m2_kspim"], uniform["m2_kspip"])
strong = strong_phase_parameters(
    np.asarray(amplitudes["A_f"]), np.asarray(amplitudes[R"\bar{A}_f"]), b_uniform
)
print(" bin      c       s    arg(c+is)   F+      F-")
for b in range(8):
    c, s = strong["c"][b], strong["s"][b]
    print(
        f"{b + 1:4d} {c:+7.3f} {s:+7.3f} {np.degrees(np.arctan2(s, c)):+9.1f}"
        f"  {strong['F_plus'][b]:.4f}  {strong['F_minus'][b]:.4f}"
    )

fig, ax = plt.subplots(figsize=(4.5, 4.5))
ax.add_patch(plt.Circle((0, 0), 1, fill=False, color="gray"))
ax.scatter(strong["c"], strong["s"])
for b in range(8):
    ax.annotate(
        str(b + 1),
        (strong["c"][b], strong["s"][b]),
        textcoords="offset points",
        xytext=(5, 5),
    )
ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect("equal")
ax.set_xlabel("$c_b$")
ax.set_ylabel("$s_b$")
plt.show()